In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import torch

from src.preprocessing import preprocess_data
from src.features import add_structural_features
from sklearn.model_selection import train_test_split

In [2]:
import transformers

print(transformers.__version__)

4.51.3


In [3]:
df = pd.read_csv("../data/train.csv")

df = preprocess_data(df)
df = add_structural_features(df)

label_map = {
    "A": 0,
    "B": 1,
    "tie": 2
}

df["target"] = df["label"].map(label_map)

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=714,
    stratify=df["target"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=714,
    stratify=temp_df["target"]
)

In [7]:
def build_candidate_text(prompt_turns, response_turns):
    dialogue = []

    for prompt, response in zip(
        prompt_turns,
        response_turns
    ):
        if isinstance(prompt, str):
            dialogue.append("User:")
            dialogue.append(prompt)

        dialogue.append("Assistant:")

        if isinstance(response, str):
            dialogue.append(response)
        else:
            dialogue.append("No response.")

    return " ".join(dialogue)

In [8]:
for data in [train_df, val_df]:
    data["candidate_a_text"] = data.apply(
        lambda row: build_candidate_text(
            row["prompt_list"],
            row["response_a_list"]
        ),
        axis=1
    )

    data["candidate_b_text"] = data.apply(
        lambda row: build_candidate_text(
            row["prompt_list"],
            row["response_b_list"]
        ),
        axis=1
    )

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

In [10]:
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "distilbert-base-multilingual-cased"
MAX_LEN = 128
TRANSFORMER_BATCH_SIZE = 16

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME)

encoder = encoder.to(device)
encoder.eval()

DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(119547, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSdpaAttention(
          (dropout): Dropout(p=0.1, inplace=False)
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): 

In [11]:
for param in encoder.parameters():
    param.requires_grad = False

In [12]:
transformer_train_df = train_df

# CPU quick test
#transformer_train_df = train_df.sample(
#    n=500,
#    random_state=714
#)

#print(transformer_train_df.shape)

(500, 27)


In [13]:
def encode_head_tail(text):
    token_ids = tokenizer.encode(
        text,
        add_special_tokens=False
    )

    body_len = MAX_LEN - 2

    if len(token_ids) > body_len:
        head_len = body_len // 2
        tail_len = body_len - head_len

        token_ids = (
            token_ids[:head_len]
            + token_ids[-tail_len:]
        )

    input_ids = (
        [tokenizer.cls_token_id]
        + token_ids
        + [tokenizer.sep_token_id]
    )

    attention_mask = [1] * len(input_ids)

    pad_len = MAX_LEN - len(input_ids)

    input_ids += (
        [tokenizer.pad_token_id] * pad_len
    )

    attention_mask += [0] * pad_len

    return input_ids, attention_mask

In [14]:
from torch.utils.data import Dataset, DataLoader


class TransformerPairDataset(Dataset):
    def __init__(self, df):
        self.a_texts = df["candidate_a_text"].tolist()
        self.b_texts = df["candidate_b_text"].tolist()
        self.labels = df["target"].values

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        a_ids, a_mask = encode_head_tail(
            self.a_texts[idx]
        )

        b_ids, b_mask = encode_head_tail(
            self.b_texts[idx]
        )

        return {
            "a_ids": torch.tensor(
                a_ids,
                dtype=torch.long
            ),
            "a_mask": torch.tensor(
                a_mask,
                dtype=torch.long
            ),
            "b_ids": torch.tensor(
                b_ids,
                dtype=torch.long
            ),
            "b_mask": torch.tensor(
                b_mask,
                dtype=torch.long
            ),
            "label": torch.tensor(
                self.labels[idx],
                dtype=torch.long
            )
        }

In [15]:
transformer_train_dataset = TransformerPairDataset(
    transformer_train_df
)

transformer_val_dataset = TransformerPairDataset(
    val_df
)

transformer_train_loader = DataLoader(
    transformer_train_dataset,
    batch_size=TRANSFORMER_BATCH_SIZE,
    shuffle=False
)

transformer_val_loader = DataLoader(
    transformer_val_dataset,
    batch_size=TRANSFORMER_BATCH_SIZE,
    shuffle=False
)

In [16]:
def extract_transformer_embeddings(
    encoder,
    data_loader,
    device
):
    all_a_embeddings = []
    all_b_embeddings = []
    all_labels = []

    encoder.eval()

    with torch.no_grad():

        for batch_idx, batch in enumerate(data_loader):

            a_ids = batch["a_ids"].to(device)
            a_mask = batch["a_mask"].to(device)

            b_ids = batch["b_ids"].to(device)
            b_mask = batch["b_mask"].to(device)

            combined_ids = torch.cat(
                [a_ids, b_ids],
                dim=0
            )

            combined_mask = torch.cat(
                [a_mask, b_mask],
                dim=0
            )

            outputs = encoder(
                input_ids=combined_ids,
                attention_mask=combined_mask
            )

            cls_embeddings = (
                outputs.last_hidden_state[:, 0, :]
            )

            batch_size = a_ids.size(0)

            h_a = cls_embeddings[:batch_size]
            h_b = cls_embeddings[batch_size:]

            all_a_embeddings.append(
                h_a.cpu()
            )

            all_b_embeddings.append(
                h_b.cpu()
            )

            all_labels.append(
                batch["label"]
            )

            if (batch_idx + 1) % 100 == 0:
                print(
                    f"Batch {batch_idx + 1}/"
                    f"{len(data_loader)}"
                )

    return (
        torch.cat(all_a_embeddings),
        torch.cat(all_b_embeddings),
        torch.cat(all_labels)
    )

In [17]:
train_h_a, train_h_b, transformer_y_train = (
    extract_transformer_embeddings(
        encoder,
        transformer_train_loader,
        device
    )
)

Token indices sequence length is longer than the specified maximum sequence length for this model (868 > 512). Running this sequence through the model will result in indexing errors


In [18]:
val_h_a, val_h_b, transformer_y_val = (
    extract_transformer_embeddings(
        encoder,
        transformer_val_loader,
        device
    )
)

Batch 100/360
Batch 200/360
Batch 300/360


In [19]:
print(train_h_a.shape)
print(train_h_b.shape)
print(val_h_a.shape)

torch.Size([500, 768])
torch.Size([500, 768])
torch.Size([5748, 768])


In [21]:
import torch.nn as nn

class TransformerPairClassifier(nn.Module):
    def __init__(
        self,
        hidden_size=768
    ):
        super().__init__()

        self.classifier = nn.Sequential(
            nn.Linear(
                hidden_size * 4,
                128
            ),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 3)
        )

    def forward(
        self,
        h_a,
        h_b
    ):
        combined = torch.cat(
            [
                h_a,
                h_b,
                h_a - h_b,
                torch.abs(h_a - h_b)
            ],
            dim=1
        )

        return self.classifier(combined)

In [22]:
class EmbeddingPairDataset(Dataset):
    def __init__(
        self,
        h_a,
        h_b,
        labels
    ):
        self.h_a = h_a.float()
        self.h_b = h_b.float()
        self.labels = labels.long()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            self.h_a[idx],
            self.h_b[idx],
            self.labels[idx]
        )

In [23]:
embedding_train_dataset = EmbeddingPairDataset(
    train_h_a,
    train_h_b,
    transformer_y_train
)

embedding_val_dataset = EmbeddingPairDataset(
    val_h_a,
    val_h_b,
    transformer_y_val
)

In [24]:
classifier_train_loader = DataLoader(
    embedding_train_dataset,
    batch_size=128,
    shuffle=True
)

classifier_val_loader = DataLoader(
    embedding_val_dataset,
    batch_size=128,
    shuffle=False
)

In [25]:
transformer_classifier = (
    TransformerPairClassifier(
        hidden_size=train_h_a.shape[1]
    ).to(device)
)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    transformer_classifier.parameters(),
    lr=1e-3
)

In [26]:
def train_transformer_classifier(
    model,
    data_loader,
    criterion,
    optimizer,
    device
):
    model.train()

    total_loss = 0

    for h_a, h_b, labels in data_loader:
        h_a = h_a.to(device)
        h_b = h_b.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(
            h_a,
            h_b
        )

        loss = criterion(
            logits,
            labels
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(data_loader)

In [27]:
def evaluate_transformer_classifier(
    model,
    data_loader,
    device
):
    model.eval()

    all_probs = []
    all_labels = []

    with torch.no_grad():

        for h_a, h_b, labels in data_loader:
            h_a = h_a.to(device)
            h_b = h_b.to(device)

            logits = model(
                h_a,
                h_b
            )

            probs = torch.softmax(
                logits,
                dim=1
            )

            all_probs.append(
                probs.cpu().numpy()
            )

            all_labels.append(
                labels.numpy()
            )

    probs = np.vstack(all_probs)
    labels = np.concatenate(all_labels)

    loss = log_loss(
        labels,
        probs,
        labels=[0, 1, 2]
    )

    return loss, probs

In [29]:
from sklearn.metrics import log_loss

EPOCHS = 5

best_val_loss = float("inf")
best_classifier_state = None

for epoch in range(EPOCHS):

    train_loss = train_transformer_classifier(
        transformer_classifier,
        classifier_train_loader,
        criterion,
        optimizer,
        device
    )

    val_loss, _ = evaluate_transformer_classifier(
        transformer_classifier,
        classifier_val_loader,
        device
    )

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Validation Log Loss: {val_loss:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        best_classifier_state = {
            key: value.cpu().clone()
            for key, value
            in transformer_classifier.state_dict().items()
        }

Epoch 1/5 | Train Loss: 1.0999 | Validation Log Loss: 1.0955
Epoch 2/5 | Train Loss: 1.0740 | Validation Log Loss: 1.0929
Epoch 3/5 | Train Loss: 1.0700 | Validation Log Loss: 1.1015
Epoch 4/5 | Train Loss: 1.0593 | Validation Log Loss: 1.1006
Epoch 5/5 | Train Loss: 1.0539 | Validation Log Loss: 1.0960


In [30]:
transformer_classifier.load_state_dict(
    best_classifier_state
)

transformer_val_logloss, transformer_val_probs = (
    evaluate_transformer_classifier(
        transformer_classifier,
        classifier_val_loader,
        device
    )
)

print(
    f"Best Transformer Validation Log Loss: "
    f"{transformer_val_logloss:.4f}"
)

Best Transformer Validation Log Loss: 1.0929
